# 10. Meaning-carrying words and metric fairness

Where are the remaining errors, and does WER treat spelling fairly?

In [1]:
from nbutil import *
import pandas as pd

## Content words and function words
WER, content-word error rate and function-word error rate (%), per system and test set.

In [2]:
M = pd.read_csv(ROOT / "results/meaning_errors/summary.csv", encoding="utf-8-sig")
for col in ("WER", "CWER", "FWER"):
    t = M.pivot(index="system", columns="set", values=col)
    display(Markdown("**" + col + "**"))
    display(t.reindex(columns=list(SETS)).rename(columns=SETS).round(2))

**WER**

set,DL Sprint eval,FLEURS,Common Voice,Kathbath
system,,,,
Hishab,21.18,14.24,7.72,8.32
IndicConformer CTC,28.13,16.03,14.04,11.20
M1,11.34,18.34,12.80,13.48
final,11.18,15.60,10.13,10.28
list correction,17.58,20.23,NaN,15.22
zero-shot,22.70,21.54,14.80,14.81


**CWER**

set,DL Sprint eval,FLEURS,Common Voice,Kathbath
system,,,,
Hishab,24.14,14.99,8.32,8.64
IndicConformer CTC,30.66,17.92,14.99,11.85
M1,13.63,19.79,13.46,13.71
final,13.47,16.94,10.69,10.62
list correction,20.49,21.14,NaN,15.38
zero-shot,23.53,21.54,14.51,14.20


**FWER**

set,DL Sprint eval,FLEURS,Common Voice,Kathbath
system,,,,
Hishab,12.34,4.29,4.21,3.04
IndicConformer CTC,21.34,4.33,5.68,3.95
M1,4.05,3.23,4.55,3.60
final,4.11,3.82,4.29,3.14
list correction,6.94,3.65,NaN,3.32
zero-shot,15.07,7.45,9.40,6.34


In [3]:
pd.read_csv(ROOT / "results/meaning_errors/predictions.csv", encoding="utf-8-sig").round(3)

,prediction,set,contrast,diff,lo,hi
0,P27a,splits_fa-eval,final CWER - final WER,2.291,1.985,2.649
1,P27a,splits_fa-eval,final CWER - final ref_rate,3.891,3.446,4.407
2,P27b,splits_fa-eval,relative gain CWER - relative gain WER,-0.080,-0.091,-0.069
3,P27a,fleurs-test,final CWER - final WER,1.342,0.874,1.789
4,P27a,fleurs-test,final CWER - final ref_rate,3.498,3.131,3.882
5,P27b,fleurs-test,relative gain CWER - relative gain WER,-0.062,-0.081,-0.045
6,P27a,cv-test,final CWER - final WER,0.555,0.449,0.658
7,P27a,cv-test,final CWER - final ref_rate,1.397,1.305,1.483
8,P27b,cv-test,relative gain CWER - relative gain WER,-0.052,-0.058,-0.046
9,P27a,kathbath-valid,final CWER - final WER,0.343,0.177,0.535


## Does the metric charge a spelling variant less than a wrong word?
Share of spelling-versus-different and dialect-versus-different pair combinations where the first costs strictly less.

In [4]:
mr = pd.read_csv(ROOT / "results/robustness/metric_robustness.csv", encoding="utf-8-sig")
for c in ("robust", "robust_lo", "robust_hi", "tie", "reversed"):
    mr[c] = (100 * mr[c]).round(1)
mr

,metric,comparison,n_first,n_different,robust,robust_lo,robust_hi,tie,reversed
0,wer,spelling_vs_different,381,164,0.0,0.0,0.0,100.0,0.0
1,wer,dialect_vs_different,197,164,0.0,0.0,0.0,100.0,0.0
2,cer,spelling_vs_different,381,164,68.6,63.8,73.3,10.2,21.3
3,cer,dialect_vs_different,197,164,57.3,51.3,63.3,11.4,31.3
4,spelling_aware_wer,spelling_vs_different,381,164,55.6,50.5,60.5,43.4,1.0
5,spelling_aware_wer,dialect_vs_different,197,164,0.5,0.0,1.5,97.1,2.4
6,deep_key,spelling_vs_different,381,164,66.8,61.7,71.4,32.1,1.1
7,deep_key,dialect_vs_different,197,164,1.5,0.0,3.4,94.9,3.6
8,uzzaman04_key,spelling_vs_different,381,164,74.4,69.5,79.1,24.1,1.4
9,uzzaman04_key,dialect_vs_different,197,164,15.5,10.9,20.2,78.4,6.1


**Conclusion:** fine-tuning removes function-word errors far more than content-word errors. The remaining errors
carry meaning. WER cannot tell a spelling variant from a wrong word. Spelling-aware WER almost never charges a
spelling variant more than a wrong word. No metric handles dialect.